In [16]:
# CELLA 1: Installa Streamlit, i motori per Excel/Parquet, la libreria Gemini e scarica Cloudflare
!pip install -q streamlit openpyxl pyarrow google-generativeai
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 118246 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.5.2) over (2026.5.2) ...
Setting up cloudflared (2026.5.2) ...
Processing triggers for man-db (2.10.2-1) ...


In [17]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import google.generativeai as genai
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    r2_score, mean_absolute_error, mean_squared_error
)

# Configurazione della pagina
st.set_page_config(page_title="Piattaforma Universale", layout="wide")

st.title("📊 Piattaforma Universale di Data Analysis & Machine Learning")
st.write(
    "Carica un dataset in qualsiasi formato (CSV, Excel, JSON, Parquet, "
    "TXT, TSV) per analizzarlo in modo interattivo e dinamico."
)

# Inserimento della API Key globale nel menu laterale
gemini_key = st.sidebar.text_input(
    "Inserisci Google Gemini API Key (Opzionale):",
    type="password"
)

# Funzione per mappare gli indici delle colonne in lettere Excel
def get_excel_col_letter(col_idx):
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    if col_idx < 26: return letters[col_idx]
    return letters[(col_idx // 26) - 1] + letters[col_idx % 26]

# Funzione universale per caricare i file con AUTO-RILEVAMENTO
def load_any_dataset(file):
    fn = file.name.lower()
    try:
        if fn.endswith('.csv'):
            bytes_sample = file.read(2048)
            file.seek(0)
            try: sample_text = bytes_sample.decode('utf-8')
            except: sample_text = bytes_sample.decode('latin-1', errors='ignore')
            sep = ';' if ';' in sample_text else '\t' if '\t' in sample_text else ','
            return pd.read_csv(file, sep=sep)
        elif fn.endswith(('.xlsx', '.xls')): return pd.read_excel(file)
        elif fn.endswith('.json'): return pd.read_json(file)
        elif fn.endswith('.parquet'): return pd.read_parquet(file)
        elif fn.endswith(('.txt', '.tsv')):
            return pd.read_csv(file, sep=None, engine='python')
        return None
    except Exception as e:
        st.error(f"Errore di caricamento: {e}")
        return None

# Inizializzazione dello stato della sessione
if 'df_cleaned' not in st.session_state: st.session_state.df_cleaned = None
if 'cleaning_confirmed' not in st.session_state: st.session_state.cleaning_confirmed = False
if 'last_uploaded_file' not in st.session_state: st.session_state.last_uploaded_file = None
if 'chat_messages' not in st.session_state: st.session_state.chat_messages = []
if 'num_cols' not in st.session_state: st.session_state.num_cols = []
if 'cat_cols' not in st.session_state: st.session_state.cat_cols = []

uploaded_file = st.file_uploader(
    "Carica il tuo file (CSV, Excel, JSON, Parquet, TXT, TSV):",
    type=["csv", "xlsx", "xls", "json", "parquet", "txt", "tsv"]
)

if uploaded_file is not None:
    if st.session_state.last_uploaded_file != uploaded_file.name:
        parsed_df = load_any_dataset(uploaded_file)
        if parsed_df is not None:
            st.session_state.df_raw = parsed_df
            st.session_state.df_cleaned = None
            st.session_state.cleaning_confirmed = False
            st.session_state.last_uploaded_file = uploaded_file.name
            st.session_state.chat_messages = []
        else:
            st.stop()

    df_raw = st.session_state.df_raw
    st.success(f"File '{uploaded_file.name}' caricato!")

    # Menu laterale
    step = st.sidebar.radio(
        "Seleziona la fase:",
        [
            "0. Presentazione & Spiegazione",
            "1. Pulizia dei Dati (Interattiva)",
            "2. Esplorazione (EDA)",
            "3. Visualizzazione Grafica",
            "4. Modello Predittivo (ML)",
            "5. Excel/Sheets & Strategie KPI",
            "6. Mitigazione Rischi & Rischio Economico",
            "7. Assistente Virtuale (AI Chat)"
        ]
    )

    # ================= FASE 0: PRESENTAZIONE =================
    if step == "0. Presentazione & Spiegazione":
        st.header("📘 Presentazione del Dataset")
        n_righe, n_colonne = df_raw.shape
        col_num = df_raw.select_dtypes(include=[np.number]).columns.tolist()
        col_testo = df_raw.select_dtypes(exclude=[np.number]).columns.tolist()
        tot_nulli = df_raw.isnull().sum().sum()

        c1, c2 = st.columns(2)
        with c1:
            st.subheader("📊 Struttura dei Dati")
            st.write(f"- **Numero totale di righe (record):** {n_righe}")
            st.write(f"- **Numero totale di colonne (variabili):** {n_colonne}")
            st.write(f"- **Variabili quantitative (numeriche):** {len(col_num)} ({', '.join(col_num[:4])}...)")
            st.write(f"- **Variabili qualitative (testo):** {len(col_testo)} ({', '.join(col_testo[:4])}...)")
        with c2:
            st.subheader("⚠️ Stato di integrità")
            if tot_nulli > 0: st.warning(f"Nel dataset sono presenti **{tot_nulli} celle vuote**. Gestiscile nella Fase 1.")
            else: st.success("Ottimo! Il dataset non presenta valori mancanti rilevati.")
        st.markdown("---")
        st.subheader("🧠 Spiegazione del Modello di Machine Learning")
        st.write("Per questo esercizio utilizzeremo l'algoritmo **Random Forest** (Foresta Decisionale Casuale). Il modello costruisce in background numerosi 'alberi decisionali' indipendenti e unisce le loro risposte per formulare la previsione finale basandosi sul voto di maggioranza.")
        st.info("💡 Spostati ora alla scheda **'1. Pulizia dei Dati (Interattiva)'** nel menu a sinistra.")

    # ================= FASE 1: PULIZIA INTERATTIVA CON MOTORE DI CONSIGLI =================
    elif step == "1. Pulizia dei Dati (Interattiva)":
        st.header("🧹 Pulizia Interattiva dei Dati & Definizione Schema")
        st.write("Configura i parametri di pulizia e classifica le singole colonne basandoti sulle raccomandazioni.")

        # --- SEZIONE CONSIGLI SCHEMA ---
        st.subheader("💡 Consigli dell'Analista sullo Schema (Tipi di Variabili)")
        st.write("Ecco la classificazione ottimale consigliata per massimizzare la precisione dei modelli:")

        auto_numeric = df_raw.select_dtypes(include=[np.number]).columns.tolist()
        schema_recommendations, suggested_numeric_defaults = [], []

        for col in df_raw.columns:
            unique_count = df_raw[col].nunique()
            is_num_native = col in auto_numeric
            if is_num_native:
                if unique_count < 10:
                    schema_recommendations.append(
                        f"- 🔸 **'{col}'**: È numerica ma ha solo **{unique_count} valori distinti**. "
                        "Ti consiglio di trattarla come CATEGORICA (Ordinale)."
                    )
                else:
                    schema_recommendations.append(
                        f"- 🔹 **'{col}'**: Ha **{unique_count} valori**. "
                        "Ti consiglio di lasciarla impostata come NUMERICA."
                    )
                    suggested_numeric_defaults.append(col)
            else:
                schema_recommendations.append(
                    f"- 🔸 **'{col}'**: È una variabile testuale con **{unique_count} categorie**. "
                    "Lasciala impostata come CATEGORICA."
                )

        for rec in schema_recommendations: st.write(rec)

        st.markdown("---")
        st.subheader("📁 1. Definizione dello Schema delle Colonne")
        selected_numeric = st.multiselect("Seleziona le singole colonne da trattare come NUMERICHE (Valori Quantitativi):", options=df_raw.columns.tolist(), default=suggested_numeric_defaults)
        selected_categorical = [col for col in df_raw.columns if col not in selected_numeric]
        st.write("Le rimanenti colonne verranno trattate come **CATEGORICHE (Testo/Scritte)**:", selected_categorical)

        st.markdown("---")
        st.subheader("📋 2. Analisi delle Anomalie e Raccomandazioni di Pulizia:")
        suggested_drops, impute_cols, rec_data = [], [], []
        for col in df_raw.columns:
            null_count = df_raw[col].isnull().sum()
            null_pct = null_count / len(df_raw)
            unique_pct = df_raw[col].nunique() / len(df_raw)

            if null_pct > 0.5:
                consiglio = "⚠️ Rimuovere (Troppi dati mancanti)"
                suggested_drops.append(col)
            elif col in selected_categorical and unique_pct > 0.8:
                consiglio = "⚠️ Rimuovere (Probabile ID o Nome, troppi valori univoci)"
                suggested_drops.append(col)
            elif null_count > 0:
                impute_cols.append(col)
                consiglio = "🔧 Mantieni e imputa"
            else:
                consiglio = "✅ Mantieni (Colonna integra)"

            rec_data.append({"Colonna": col, "Tipo Impostato": "Numerica" if col in selected_numeric else "Categorica", "Celle Vuote": f"{null_count} ({null_pct:.1%})", "Raccomandazione": consiglio})

        st.table(pd.DataFrame(rec_data))
        st.markdown("---")

        cols_to_drop = st.multiselect("Seleziona quali singole colonne vuoi ELIMINARE dal dataset:", options=df_raw.columns.tolist(), default=suggested_drops)
        remaining_cols_with_nulls = [col for col in df_raw.columns if col not in cols_to_drop and df_raw[col].isnull().sum() > 0]
        imputation_strategy = "Sostituzione consigliata"

        if remaining_cols_with_nulls:
            st.write("Nelle colonne rimenenti ci sono dei valori mancanti:")
            imputation_strategy = st.radio("Come vuoi gestire questi valori vuoti?", ["Sostituzione consigliata (Mediana per numeri, Moda per testi)", "Elimina l'intera riga per ogni valore mancante (Dropna)", "Sostituisci i numeri con 0 e i testi con 'Sconosciuto'"])

        if st.button("Applica e Conferma questa Pulizia"):
            df_temp = df_raw.drop(columns=cols_to_drop)
            if remaining_cols_with_nulls:
                if imputation_strategy == "Elimina l'intera riga per ogni valore mancante (Dropna)":
                    df_temp = df_temp.dropna()
                elif imputation_strategy == "Sostituisci i numeri con 0 e i testi con 'Sconosciuto'":
                    for col in df_temp.columns:
                        if col in selected_numeric: df_temp[col] = df_temp[col].fillna(0)
                        else: df_temp[col] = df_temp[col].fillna("Sconosciuto")
                else:
                    for col in df_temp.columns:
                        if df_temp[col].isnull().sum() > 0:
                            if col in selected_numeric: df_temp[col] = df_temp[col].fillna(df_temp[col].median())
                            else: df_temp[col] = df_temp[col].fillna(df_temp[col].mode()[0])

            st.session_state.df_cleaned = df_temp
            st.session_state.num_cols = [c for c in selected_numeric if c in df_temp.columns]
            st.session_state.cat_cols = [c for c in selected_categorical if c in df_temp.columns]
            st.session_state.cleaning_confirmed = True
            st.balloons()
            st.success("Pulizia e schema applicati correttamente!")

        if st.session_state.cleaning_confirmed and st.session_state.df_cleaned is not None:
            st.subheader("Anteprima del dataset pulito:")
            st.dataframe(st.session_state.df_cleaned.head(5))

        st.markdown("---")
        show_code = st.checkbox("💻 Mostra il codice Python universale per la pulizia")
        if show_code:
            code_cleaning = """import pandas as pd

# 1. Carica il dataset originale
df = pd.read_csv("tuo_file.csv")

# 2. Specifica le colonne da eliminare (scelte dall'utente)
cols_to_drop = ['Colonna_ID', 'Colonna_Testo_Inutile']
df_cleaned = df.drop(columns=cols_to_drop)

# 3. Gestisci i valori mancanti nelle colonne rimanenti (Imputazione consigliata)
for col in df_cleaned.columns:
    if df_cleaned[col].isnull().sum() > 0:
        if df_cleaned[col].dtype in ['int64', 'float64']:
            df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())
        else:
            df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])

print("Dati puliti con successo! Nuove dimensioni:", df_cleaned.shape)
"""
            st.code(code_cleaning, language="python")

    # Controllo di sicurezza
    elif not st.session_state.cleaning_confirmed:
        st.warning("⚠️ Completa prima la scheda **'1. Pulizia dei Dati (Interattiva)'**.")

    # ================= FASE 2: EDA =================
    elif step == "2. Esplorazione (EDA)":
        df_cleaned = st.session_state.df_cleaned
        st.header("📊 Esplorazione del Dataset (EDA)")
        st.write(df_cleaned.describe(include='all'))

        explore_col = st.selectbox("Seleziona una colonna per analizzarne la distribuzione:", df_cleaned.columns)
        c1, c2 = st.columns(2)
        with c1: st.write(df_cleaned[explore_col].value_counts())
        with c2: st.write(df_cleaned.dtypes.to_frame(name='Tipo di Dato'))

        st.markdown("---")
        show_code = st.checkbox("💻 Mostra il codice Python per l'esplorazione dei dati")
        if show_code:
            code_eda = (
                "import pandas as pd\n\n"
                "df = pd.read_csv('dati_puliti.csv')\n"
                "print(df.describe(include='all'))\n"
                "print(df.dtypes)\n"
                f"print(df['{explore_col}'].value_counts())"
            )
            st.code(code_eda, language="python")

    # ================= FASE 3: VISUALIZZAZIONE GRAFICA RISPETTANDO LO SCHEMA UTENTE =================
    elif step == "3. Visualizzazione Grafica":
        df_cleaned = st.session_state.df_cleaned
        col_num = st.session_state.num_cols
        col_cat = st.session_state.cat_cols
        col_tutte = df_cleaned.columns.tolist()

        st.header("📈 Visualizzazione Grafica Avanzata")
        st.subheader("🔍 Quali sono i grafici migliori per questo Dataset?")

        # Consigli intelligenti basati sullo schema definito dall'utente
        is_financial = any(x in ["price", "fare", "revenue", "amount", "sales", "cost", "prezzo", "ricavi", "tariffa", "importo", "vendit", "spesa"] for x in [c.lower() for c in col_num])
        is_risk = any(x in ["survived", "survival", "death", "mort", "sopravv", "churn", "abbandono", "fail", "guast"] for x in [c.lower() for c in col_tutte])

        if is_financial: st.write("- 💡 **Box Plot**: Rilevate colonne finanziarie, ideale per identificare subito spese o importi anomali.")
        if is_risk: st.write("- 💡 **Violin Plot**: Rilevate variabili di rischio, ideale per vedere la distribuzione dei valori tra classi.")
        if len(col_num) > 1: st.write("- 💡 **Mappa delle Correlazioni**: Consigliata per scoprire relazioni nascoste.")

        st.markdown("---")
        tipo_grafico = st.selectbox("Seleziona il tipo di grafico:", ["Mappa di Correlazione (Heatmap)", "Scatter Plot", "Box Plot (Rilevatore Outlier)", "Violin Plot", "Line Plot"])

        # 1. HEATMAP
        if tipo_grafico == "Mappa di Correlazione (Heatmap)":
            st.subheader("Mappa delle Correlazioni di Seaborn")
            if len(col_num) > 1:
                fig, ax = plt.subplots(figsize=(8, 5))
                sns.heatmap(df_cleaned[col_num].corr(), annot=True, cmap="coolwarm", fmt=".2f", ax=ax)
                st.pyplot(fig)

                corr_m = df_cleaned[col_num].corr().abs()
                np.fill_diagonal(corr_m.values, 0)
                max_val = corr_m.max().max()
                if max_val > 0.5:
                    max_cols = corr_m.unstack().idxmax()
                    st.info(f"📝 **Commento dell'Analista:** La correlazione più forte è tra **{max_cols[0]}** e **{max_cols[1]}** ({max_val:.2f}).")
                else:
                    st.info("📝 **Commento dell'Analista:** Le variabili quantitative hanno una correlazione lineare debole tra loro.")

                st.markdown("---")
                show_code = st.checkbox("💻 Mostra codice Python per la Heatmap")
                if show_code:
                    code_heatmap = "import matplotlib.pyplot as plt\nimport seaborn as sns\n\ndf_numeric = df.select_dtypes(include=['int64', 'float64'])\nplt.figure(figsize=(8, 5))\nsns.heatmap(df_numeric.corr(), annot=True, cmap='coolwarm', fmt='.2f')\nplt.title('Mappa delle Correlazioni')\nplt.show()"
                    st.code(code_heatmap, language="python")
            else:
                st.warning("Devi impostare almeno due colonne come numeriche nella Fase 1 per calcolare la correlazione.")

        # 2. SCATTER PLOT
        elif tipo_grafico == "Scatter Plot":
            st.subheader("Scatter Plot (Seaborn)")
            if len(col_num) > 1:
                c1, c2, c3 = st.columns(3)
                with c1: x_col = st.selectbox("Asse X (Numerica):", col_num)
                with c2: y_col = st.selectbox("Asse Y (Numerica):", [c for c in col_num if c != x_col])
                with c3: h_col = st.selectbox("Colore (Opzionale):", ["Nessuno"] + col_tutte)

                fig, ax = plt.subplots(figsize=(8, 5))
                sns.scatterplot(data=df_cleaned, x=x_col, y=y_col, hue=None if h_col == "Nessuno" else h_col, ax=ax)
                st.pyplot(fig)

                r_val = df_cleaned[x_col].corr(df_cleaned[y_col])
                forza = "forte" if abs(r_val) > 0.7 else "moderata" if abs(r_val) > 0.4 else "debole o assente"
                st.info(f"📝 **Commento dell'Analista:** Il coefficiente r di Pearson tra '{x_col}' e '{y_col}' è pari a **{r_val:.2f}** ({forza}).")

                st.markdown("---")
                show_code = st.checkbox("💻 Mostra codice Python per lo Scatter Plot")
                if show_code:
                    hue_code = f", hue='{h_col}'" if h_col != "Nessuno" else ""
                    code_scatter = f"import matplotlib.pyplot as plt\nimport seaborn as sns\n\nplt.figure(figsize=(8, 5))\nsns.scatterplot(data=df, x='{x_col}', y='{y_col}'{hue_code})\nplt.title('Scatter Plot')\nplt.show()"
                    st.code(code_scatter, language="python")
            else:
                st.warning("Configura più colonne numeriche nello schema della Fase 1.")

        # 3. BOX PLOT
        elif tipo_grafico == "Box Plot (Rilevatore Outlier)":
            st.subheader("Box Plot (Seaborn)")
            if len(col_num) > 0:
                c1, c2 = st.columns(2)
                with c1: y_box = st.selectbox("Analizza (Numerica):", col_num)
                with c2: x_box = st.selectbox("Raggruppa (Opzionale):", ["Nessuna"] + col_tutte)
                fig, ax = plt.subplots(figsize=(8, 5))
                sns.boxplot(data=df_cleaned, x=None if x_box == "Nessuna" else x_box, y=y_box, ax=ax, palette="pastel")
                st.pyplot(fig)

                q1 = df_cleaned[y_box].quantile(0.25)
                q3 = df_cleaned[y_box].quantile(0.75)
                iqr = q3 - q1
                outliers = df_cleaned[(df_cleaned[y_box] < (q1 - 1.5 * iqr)) | (df_cleaned[y_box] > (q3 + 1.5 * iqr))]
                st.info(f"📝 **Commento dell'Analista:** La colonna '{y_box}' presenta **{len(outliers)} anomalie (outliers)** calcolate tramite l'Intervallo Interquartile (IQR).")

                st.markdown("---")
                show_code = st.checkbox("💻 Mostra codice Python per il Box Plot")
                if show_code:
                    x_code = f"x='{x_box}', " if x_box != "Nessuna" else ""
                    code_box = f"import matplotlib.pyplot as plt\nimport seaborn as sns\n\nplt.figure(figsize=(8, 5))\nsns.boxplot(data=df, {x_code}y='{y_box}', palette='pastel')\nplt.title('Box Plot')\nplt.show()"
                    st.code(code_box, language="python")
            else:
                st.warning("Configura almeno una colonna numerica nella Fase 1.")

        # 4. VIOLIN PLOT
        elif tipo_grafico == "Violin Plot":
            st.subheader("Violin Plot (Seaborn)")
            if len(col_num) > 0:
                c1, c2 = st.columns(2)
                with c1: y_v = st.selectbox("Variabile (Numerica):", col_num)
                with c2: x_v = st.selectbox("Raggruppa (Opzionale):", ["Nessuno"] + col_tutte)
                fig, ax = plt.subplots(figsize=(8, 5))
                sns.violinplot(data=df_cleaned, x=None if x_v == "Nessuno" else x_v, y=y_v, ax=ax, palette="muted")
                st.pyplot(fig)
                st.info("📝 **Commento dell'Analista:** La larghezza della curva indica la densità delle osservazioni.")

                st.markdown("---")
                show_code = st.checkbox("💻 Mostra codice Python per il Violin Plot")
                if show_code:
                    x_code = f"x='{x_v}', " if x_v != "Nessuno" else ""
                    code_violin = f"import matplotlib.pyplot as plt\nimport seaborn as sns\n\nplt.figure(figsize=(8, 5))\nsns.violinplot(data=df, {x_code}y='{y_v}', palette='muted')\nplt.title('Violin Plot')\nplt.show()"
                    st.code(code_violin, language="python")
            else:
                st.warning("Configura almeno una colonna numerica nella Fase 1.")

        # 5. LINE PLOT
        elif tipo_grafico == "Line Plot":
            st.subheader("Line Plot (Seaborn)")
            if len(col_num) > 1:
                c1, c2 = st.columns(2)
                with c1: x_l = st.selectbox("Asse X (Numerica):", col_num)
                with c2: y_l = st.selectbox("Asse Y (Numerica):", [c for c in col_num if c != x_l])
                fig, ax = plt.subplots(figsize=(8, 5))
                sns.lineplot(data=df_cleaned.sort_values(by=x_l), x=x_l, y=y_l, ax=ax, marker="o")
                st.pyplot(fig)
                st.info(f"📝 **Commento dell'Analista:** Questo grafico mostra l'andamento sequenziale di '{y_l}' rispetto a '{x_l}'.")

                st.markdown("---")
                show_code = st.checkbox("💻 Mostra codice Python per il Line Plot")
                if show_code:
                    code_line = f"import matplotlib.pyplot as plt\nimport seaborn as sns\ndf_sorted = df.sort_values(by='{x_l}')\nplt.figure(figsize=(8, 5))\nsns.lineplot(data=df_sorted, x='{x_l}', y='{y_l}', marker='o')\nplt.title('Andamento Lineare')\nplt.show()"
                    st.code(code_line, language="python")
            else:
                st.warning("Configura più colonne numeriche nello schema della Fase 1.")

# L'applicazione continua nel file Cella 2B

Overwriting app.py


In [18]:
%%writefile -a app.py

    # ================= FASE 4: MACHINE LEARNING RISPETTANDO LO SCHEMA UTENTE =================
    elif step == "4. Modello Predittivo (ML)":
        df_cleaned = st.session_state.df_cleaned
        col_num = st.session_state.num_cols
        col_cat = st.session_state.cat_cols

        st.header("🤖 Addestramento & Previsione (Machine Learning)")
        st.write("Configura la tipologia di analisi e i parametri dell'algoritmo.")

        # Scelta del tipo di Task
        task_type = st.radio("Seleziona il tipo di analisi predittiva (Task):", ["Classificazione (Prevedere categorie)", "Regressione (Prevedere valori continui)"])
        is_classification_task = "Classificazione" in task_type

        target_col = st.selectbox("Seleziona la colonna Target da prevedere:", df_cleaned.columns, index=0)
        possible_features = [col for col in df_cleaned.columns if col != target_col]
        features_selected = st.multiselect("Seleziona i fattori predittivi (Features):", possible_features, default=possible_features[:min(len(possible_features), 5)])

        if not features_selected:
            st.warning("Seleziona almeno una colonna come predittore per procedere.")
            st.stop()

        # --- PREPARAZIONE DEI DATI ---
        df_ml = df_cleaned[features_selected + [target_col]].dropna()
        X_raw, y_raw = df_ml[features_selected].copy(), df_ml[target_col].copy()

        target_is_categorical = False
        target_mapping, reverse_target_mapping = None, None

        if is_classification_task:
            target_is_categorical = True
            unique_targets = sorted(y_raw.unique())
            target_mapping = {val: idx for idx, val in enumerate(unique_targets)}
            reverse_target_mapping = {idx: val for idx, val in enumerate(unique_targets)}
            y = y_raw.map(target_mapping)
        else:
            y = pd.to_numeric(y_raw, errors='coerce')
            valid_target_idx = y.notnull()
            X_raw, y = X_raw[valid_target_idx], y[valid_target_idx]

        # Codifica delle Features RISPETTANDO LO SCHEMA UTENTE
        X = pd.DataFrame()
        cat_encoders = {}

        for col in features_selected:
            if col in col_cat:
                unique_vals = sorted(X_raw[col].astype(str).unique())
                cat_encoders[col] = unique_vals
                mapping = {val: idx for idx, val in enumerate(unique_vals)}
                X[col] = X_raw[col].astype(str).map(mapping)
            else:
                X[col] = pd.to_numeric(X_raw[col], errors='coerce')

        if X.shape[0] == 0:
            st.error("Il dataset di addestramento è rimasto vuoto dopo la rimozione dei valori non numerici.")
            st.stop()

        # Parametri
        st.subheader("⚙️ Configura i parametri dell'algoritmo")
        col_param1, col_param2 = st.columns(2)
        with col_param1:
            n_trees = st.slider("Numero di Alberi decisionali (n_estimators):", 10, 300, 100, 10)
        with col_param2:
            max_depth_choice = st.radio("Profondità massima degli alberi (max_depth):", ["Nessun Limite", "Limitata"])
            max_depth = None if max_depth_choice == "Nessun Limite" else st.slider("Seleziona profondità:", 2, 20, 8)

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        model = RandomForestClassifier(random_state=42, n_estimators=n_trees, max_depth=max_depth) if is_classification_task else RandomForestRegressor(random_state=42, n_estimators=n_trees, max_depth=max_depth)

        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            # SEZIONE METRICHE CLASSIFICAZIONE
            if is_classification_task:
                acc = accuracy_score(y_test, y_pred)
                st.success(f"Modello Classifier addestrato! Accuratezza: **{acc:.2%}**")
                col_metrics1, col_metrics2 = st.columns(2)

                with col_metrics1:
                    st.write("Report dettagliato delle metriche:")
                    try:
                        present_classes = np.unique(np.concatenate((y_test, y_pred)))
                        labels_names = [str(reverse_target_mapping[i]) for i in present_classes]
                        report = classification_report(y_test, y_pred, labels=present_classes, target_names=labels_names, output_dict=True)
                        st.write(pd.DataFrame(report).transpose())
                    except:
                        st.warning("Seleziona un target con meno classi uniche per analizzare le metriche.")

                with col_metrics2:
                    st.write("Matrice di Confusione (Grafica):")
                    num_unique_classes = len(np.unique(y_test))
                    if num_unique_classes > 20: st.warning("La colonna selezionata ha troppe classi distinte.")
                    else:
                        fig_cm, ax_cm = plt.subplots(figsize=(5, 3.5))
                        cm = confusion_matrix(y_test, y_pred)
                        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax_cm)
                        ax_cm.set_xlabel("Previsto")
                        ax_cm.set_ylabel("Reale")
                        st.pyplot(fig_cm)

            # SEZIONE METRICHE REGRESSIONE
            else:
                r2 = r2_score(y_test, y_pred)
                mae = mean_absolute_error(y_test, y_pred)
                rmse = np.sqrt(mean_squared_error(y_test, y_pred))

                st.success(f"Modello Regressor addestrato! Coefficiente R²: **{r2:.4f}**")
                col_reg1, col_reg2 = st.columns(2)

                with col_reg1:
                    st.write("Metriche di Errore:")
                    st.write(pd.DataFrame({
                        "Metrica": ["R-squared (R²)", "MAE", "RMSE"],
                        "Valore Ottenuto": [f"{r2:.4f}", f"{mae:.4f}", f"{rmse:.4f}"],
                    }))

                with col_reg2:
                    st.write("Grafico Diagnostico: Reali vs Previsti")
                    fig_reg, ax_reg = plt.subplots(figsize=(5, 3.5))
                    sns.scatterplot(x=y_test, y=y_pred, alpha=0.6, ax=ax_reg)
                    ax_reg.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
                    st.pyplot(fig_reg)

            # SIMULATORE
            st.markdown("---")
            st.subheader("🔮 Simulatore Predittivo")
            user_inputs, cols_inputs = {}, st.columns(2)

            for i, col in enumerate(features_selected):
                with cols_inputs[i % 2]:
                    if col in cat_encoders:
                        selected_val = st.selectbox(f"Seleziona '{col}':", cat_encoders[col])
                        user_inputs[col] = cat_encoders[col].index(selected_val)
                    else:
                        min_val = float(X_raw[col].min())
                        max_val = float(X_raw[col].max())
                        mean_val = float(X_raw[col].mean())
                        user_inputs[col] = min_val if min_val == max_val else st.slider(f"Modifica '{col}':", min_val, max_val, mean_val)

            if st.button("Calcola Previsione"):
                input_df = pd.DataFrame([user_inputs])
                prediction = model.predict(input_df)[0]
                if is_classification_task:
                    probabilities = model.predict_proba(input_df)[0]
                    lbl = reverse_target_mapping[prediction] if target_is_categorical else prediction
                    st.info(f"Classe Prevista: **{lbl}** | Confidenza: {max(probabilities):.2%}")
                else:
                    st.info(f"Valore Numerico Stimato: **{prediction:.4f}**")

            st.markdown("---")
            show_code = st.checkbox("💻 Mostra il codice Python per questo modello predittivo")
            if show_code:
                if is_classification_task:
                    import_model_line = "from sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix"
                    instantiate_model_line = f"model = RandomForestClassifier(random_state=42, n_estimators={n_trees}, {depth_code})"
                    evaluation_code = "y_pred = model.predict(X_test)\nprint('Accuratezza:', accuracy_score(y_test, y_pred))"
                else:
                    import_model_line = "from sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error"
                    instantiate_model_line = f"model = RandomForestRegressor(random_state=42, n_estimators={n_trees}, {depth_code})"
                    evaluation_code = "y_pred = model.predict(X_test)\nprint('R2 Score:', r2_score(y_test, y_pred))"

                code_ml = (
                    "import pandas as pd\n"
                    "from sklearn.model_selection import train_test_split\n"
                    f"{import_model_line}\n\n"
                    "df = pd.read_csv('tuo_file.csv')\n"
                    f"y = df['{target_col}']\n"
                    "X = df.drop(columns=[target_col])\n\n"
                    "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n"
                    f"{instantiate_model_line}\n"
                    "model.fit(X_train, y_train)\n"
                    f"{evaluation_code}"
                )
                st.code(code_ml, language="python")
        except Exception as e:
            st.error(f"Errore Machine Learning: {e}")

    # ================= FASE 5: EXCEL/SHEETS & STRATEGIE KPI =================
    elif step == "5. Excel/Sheets & Strategie KPI":
        st.header("📋 Equivalenti Excel/Sheets & Strategie di Miglioramento KPI")
        tab1, tab2 = st.tabs(["💻 Formule Dinamiche Excel / Google Sheets", "🎯 Strategie e Soluzioni KPI"])

        with tab1:
            st.subheader("🧹 Formule Excel/Sheets per il TUO dataset:")
            null_cols = [col for col in df_raw.columns if df_raw[col].isnull().sum() > 0]
            if null_cols:
                excel_formulas = []
                for col in null_cols:
                    col_idx = df_raw.columns.get_loc(col)
                    letter = get_excel_col_letter(col_idx)
                    if col in st.session_state.num_cols:
                        m_v = df_raw[col].median()
                        f_it, f_en = f"=SE.ERRORE({letter}2; MEDIANA({letter}:{letter}))", f"=IFERROR({letter}2, MEDIAN({letter}:{letter}))"
                        desc = f"Riempi i vuoti in **{col}** con la mediana ({m_v:.2f})."
                    else:
                        mo_v = df_raw[col].mode()[0]
                        f_it, f_en = f'=SE.ERRORE({letter}2; "{mo_v}")', f'=IFERROR({letter}2, "{mo_v}")'
                        desc = f"Riempi i vuoti in **{col}** con la moda (\"{mo_v}\")."
                    excel_formulas.append({"Colonna": f"{col} ({letter})", "Excel (IT)": f_it, "Sheets (EN)": f_en, "Dettaglio": desc})
                st.table(pd.DataFrame(excel_formulas))
            else:
                st.success("Ottimo! Nessuna cella vuota rilevata.")

        with tab2:
            st.subheader("🎯 Strategie per migliorare i tuoi KPI:")
            c_low = [c.lower() for c in df_raw.columns]
            is_financial = any(x in ["price", "fare", "revenue", "amount", "sales", "cost", "prezzo", "ricavi", "tariffa", "importo", "vendit", "spesa"] for x in c_low)
            is_demographic = any(x in ["age", "sex", "gender", "paziente", "pax", "eta", "sesso", "patient", "population", "demog"] for x in c_low)
            is_risk = any(x in ["survived", "survival", "death", "mort", "sopravv", "churn", "abbandono", "fail", "guast"] for x in c_low)

            if is_financial:
                st.info("📊 **Dataset Rilevato: Finanziario / Vendite**")
                st.write("**1. KPI:** AOV (Carrello medio), LTV (Lifetime Value), CAC (Costo acquisizione), Margine lordo.")
                st.write("**2. Prezzi Dinamici:** Se la Fase 3 mostra forti oscillazioni di prezzo per categorie, ottimizza le tariffe in base a canali o stagionalità.")
                st.write("**3. Up-Selling:** Usa la Fase 4 per identificare i fattori (es. demografici, categoria d'acquisto) che aumentano la spesa e crea promozioni mirate.")
            elif is_risk:
                st.info("🛡️ **Dataset Rilevato: Analisi del Rischio / Churn / Anomalie**")
                st.write("**1. KPI:** Retention Rate (Tasso di fidelizzazione), Churn Rate (Abbandono), MTBF (Tempo medio tra anomalie).")
                st.write("**2. Prevenzione:** Se le Fasi 3 e 4 indicano segmenti ad alto rischio di perdita o anomalie tecniche, rialloca le risorse commerciali o manutentive per intervenire.")
                st.write("**3. Early Warning:** Collega il simulatore predittivo (Fase 4) al monitoraggio per inviare allerte automatiche al raggiungimento di soglie critiche.")
            elif is_demographic:
                st.info("👥 **Dataset Rilevato: Demografico / Clinico / Popolazione**")
                st.write("**1. KPI:** Tasso di successo per cluster, Distribuzione territoriale, Efficienza operativa per fascia d'età.")
                st.write("**2. Targeting:** Se la Fase 2 mostra disparità nelle distribuzioni dei segmenti, personalizza i canali di servizio (es. digital per giovani, assistito per anziani).")
                st.write("**3. Pianificazione:** Usa la Fase 3 per analizzare i picchi di affluenza dei vari gruppi demografici e pianificare il personale in reparto.")
            else:
                st.info("⚙️ **Dataset Rilevato: Generico / Operativo**")
                st.write("**1. KPI:** OEE (Efficienza impianti), Lead Time (Tempi di ciclo), Tasso di scarto/errore.")
                st.write("**2. Standardizzazione:** Usa l'EDA (Fase 2) per trovare i processi con maggiore variabilità e introduci procedure operative standard (SOP).")
                st.write("**3. ML Operativo:** Usa il modello predittivo (Fase 4) per anticipare colli di bottiglia o guasti imminenti nei flussi.")

    # ================= FASE 6: MITIGAZIONE RISCHI & RISCHIO ECONOMICO =================
    elif step == "6. Mitigazione Rischi & Rischio Economico":
        df_cleaned = st.session_state.df_cleaned
        st.header("🛡️ Fase 6: Mitigazione Rischi & Rischio Economico")
        st.write("Identifichiamo le principali anomalie o perdite latenti all'interno del dataset e pianifichiamo soluzioni tramite grafici predittivi dedicati.")

        # --- SEZIONE 1: SCANSIONE AUTOMATICA ANOMALIE ECONOMICHE ---
        st.subheader("🔍 1. Rilevamento delle Criticità & Outliers Economici")
        columns_lower = [c.lower() for c in df_cleaned.columns]

        # Ricerca colonne monetarie
        colonne_finanziarie = [col for col in df_cleaned.columns if any(x in col.lower() for x in ["price", "fare", "revenue", "amount", "sales", "cost", "prezzo", "ricavi", "tariffa", "importo", "vendit", "spesa"])]

        if colonne_finanziarie:
            col_fin = colonne_finanziarie[0]
            media = df_cleaned[col_fin].mean()
            std_dev = df_cleaned[col_fin].std()
            limite_anomalia = media + 3 * std_dev
            anomalie = df_cleaned[df_cleaned[col_fin] > limite_anomalia]

            if len(anomalie) > 0:
                st.warning(f"⚠️ **Rilevate Anomalie Finanziarie:** Nella colonna **{col_fin}** sono stati identificati **{len(anomalie)} record** con costi/ricavi eccezionalmente alti (superiori a {limite_anomalia:.2f}).")
                with st.expander("👁️ Visualizza i record anomali"):
                    st.dataframe(anomalie)
            else:
                st.success(f"Nessuna spesa anomala fuori controllo (Z-Score > 3) rilevata nella colonna {col_fin}.")
        else:
            st.info("Nessuna colonna finanziaria esplicita rilevata per il controllo degli outliers a 3 Sigma.")

        st.markdown("---")

        # --- SEZIONE 2: MODELLO DI RISK ANALYSIS & GRAFICI PREDITTIVI ---
        st.subheader("📈 2. Grafici Predittivi per la Riduzione del Rischio (ML-Powered)")
        risk_target = st.selectbox("Seleziona la variabile di rischio da prevenire (Target):", df_cleaned.columns)
        possible_risk_features = [col for col in df_cleaned.columns if col != risk_target]
        risk_features = st.multiselect("Seleziona i fattori di rischio da analizzare:", possible_risk_features, default=possible_risk_features[:min(len(possible_risk_features), 5)])

        if risk_features:
            df_risk_ml = df_cleaned[risk_features + [risk_target]].dropna()
            X_risk = pd.DataFrame()
            for col in risk_features:
                if df_risk_ml[col].dtype == 'object':
                    unique_vals = sorted(df_risk_ml[col].astype(str).unique())
                    mapping = {val: idx for idx, val in enumerate(unique_vals)}
                    X_risk[col] = df_risk_ml[col].astype(str).map(mapping)
                else:
                    X_risk[col] = pd.to_numeric(df_risk_ml[col], errors='coerce')

            y_risk = df_risk_ml[risk_target]
            if y_risk.dtype == 'object' or y_risk.nunique() < 10:
                unique_t = sorted(y_risk.unique())
                y_risk = y_risk.map({val: idx for idx, val in enumerate(unique_t)})
                is_classification = True
            else:
                y_risk = pd.to_numeric(y_risk, errors='coerce')
                is_classification = False

            if X_risk.shape[0] > 0:
                risk_model = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=8) if is_classification else RandomForestRegressor(random_state=42, n_estimators=100, max_depth=8)
                risk_model.fit(X_risk, y_risk)

                col_graph1, col_graph2 = st.columns(2)
                with col_graph1:
                    st.write("**A. Dove intervenire? (Feature Importance)**")
                    importances = risk_model.feature_importances_
                    indices = np.argsort(importances)[::-1]

                    df_imp = pd.DataFrame({
                        "Fattore": [risk_features[i] for i in indices],
                        "Impatto sul Rischio": [importances[i] for i in indices]
                    }).sort_values(by="Impatto sul Rischio", ascending=False)

                    fig_imp, ax_imp = plt.subplots(figsize=(5, 3.5))
                    sns.barplot(data=df_imp, x="Impatto sul Rischio", y="Fattore", palette="Reds_r", ax=ax_imp)
                    st.pyplot(fig_imp)
                    st.write("*📝 **Analisi dell'Analista:** Questo grafico predice quali variabili hanno il peso maggiore nel determinare l'anomalia o la perdita. Intervenendo prioritariamente sulla prima variabile in alto, otterrai la massima efficacia di riduzione del rischio.*")

                with col_graph2:
                    st.write("**B. Distribuzione della Probabilità di Rischio**")
                    fig_prob, ax_prob = plt.subplots(figsize=(5, 3.5))
                    if is_classification:
                        prob_predictions = risk_model.predict_proba(X_risk)[:, -1]
                        sns.kdeplot(prob_predictions, fill=True, color="red", ax=ax_prob)
                        ax_prob.set_xlim(0, 1)
                        ax_prob.set_xlabel("Probabilità di Rischio Estimata")
                    else:
                        sns.kdeplot(risk_model.predict(X_risk), fill=True, color="red", ax=ax_prob)
                        ax_prob.set_xlabel("Valore Previsto dell'Evento di Perdita")
                    st.pyplot(fig_prob)
                    st.write("*📝 **Analisi dell'Analista:** Questa curva mostra la concentrazione del rischio sulla tua popolazione di record. Picchi verso destra indicano l'urgenza di un piano di contenimento.*")

                st.markdown("---")
                st.subheader("🎯 3. Soluzioni Operative Consigliate")
                top_feature = df_imp.iloc[0]["Fattore"]
                st.write(f"- **Azione Prioritaria (Focus su '{top_feature}'):** Il modello stima che {top_feature} sia il driver principale del rischio. Devi focalizzarti unicamente su questa variabile.")
                st.write("- **Manutenzione e Controllo Preventivo:** Utilizza la curva di probabilità per identificare i record a rischio e intervieni in anticipo.")
                st.write("- **Soglia di Allarme:** Se i sensori o dati indicano valori critici calcolati nel simulatore, attiva un piano di emergenza automatico.")
            else:
                st.warning("Dati insufficienti per addestrare il modello di rischio.")
        else:
            st.info("Seleziona almeno un fattore predittivo per generare i grafici di prevenzione del rischio.")

    # ================= FASE 7: ASSISTENTE VIRTUALE (AI CHAT!) =================
    elif step == "7. Assistente Virtuale (AI Chat)":
        st.header("🤖 7. Assistente Virtuale (AI Chat)")
        st.write("Usa questo spazio per fare domande sui tuoi dati, interpretare i grafici o chiedere aiuto per la scrittura di script correttivi.")

        # Area Chat
        for message in st.session_state.chat_messages:
            with st.chat_message(message["role"]):
                st.markdown(message["content"])

        # Input utente
        if user_query := st.chat_input("Fai una domanda all'IA sul tuo dataset..."):
            with st.chat_message("user"):
                st.markdown(user_query)
            st.session_state.chat_messages.append({"role": "user", "content": user_query})

            with st.chat_message("assistant"):
                if gemini_key:
                    genai.configure(api_key=gemini_key)

                    df_info = st.session_state.df_cleaned if st.session_state.df_cleaned is not None else df_raw
                    system_prompt = (
                        f"Sei un assistente esperto in Data Analysis e Business Intelligence.\n"
                        f"L'utente ha caricato un dataset con {df_info.shape[0]} righe e {df_info.shape[1]} colonne.\n"
                        f"Le variabili del dataset e i relativi tipi di dato sono: {df_info.dtypes.to_dict()}.\n"
                        f"Fornisci analisi dettagliate, risposte professionali o codice Python se richiesto.\n\n"
                    )

                    # Ciclo di fallback dinamico per l'anno 2026
                    success = False
                    for model_name in ['gemini-3.5-flash', 'gemini-3.1-flash-lite', 'gemini-2.5-flash']:
                        try:
                            model_ai = genai.GenerativeModel(model_name)
                            response = model_ai.generate_content(system_prompt + user_query)
                            st.markdown(response.text)
                            st.session_state.chat_messages.append({"role": "assistant", "content": response.text})
                            success = True
                            break
                        except:
                            continue

                    if not success:
                        st.error("Impossibile stabilire una connessione. Verifica che la tua API Key sia corretta e attiva.")
                else:
                    df_info = st.session_state.df_cleaned if st.session_state.df_cleaned is not None else df_raw
                    demo_text = (
                        "⚠️ **Modalità Demo**: Non hai inserito una Gemini API Key nel campo in alto del menu laterale.\n\n"
                        f"Per poter chattare realmente con l'Intelligenza Artificiale applicata direttamente sul tuo dataset, "
                        f"inserisci una chiave valida (ottenibile gratuitamente su ai.google.dev).\n\n"
                        f"**Simulatore Analista Dati:** Sulla base dei dati caricati posso comunque dirti che il tuo dataset "
                        f"dispone di **{df_info.shape[1]} colonne** e che i fattori predittivi inseriti sono pronti "
                        f"per l'addestramento predittivo."
                    )
                    st.markdown(demo_text)
                    st.session_state.chat_messages.append({"role": "assistant", "content": demo_text})
else:
    st.info("In attesa del caricamento di un file per iniziare l'analisi.")

Appending to app.py


In [ ]:
# CELLA 3: Avvia l'applicazione e il tunnel di Cloudflare
!streamlit run app.py & cloudflared tunnel --url http://localhost:8501

2026-05-31T14:48:55Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-31T14:48:55Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-05-31 14:48:59.505 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.124.145.78:8501

2026-05-31T14:49:01Z INF +--------------------------------------------------------